In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
delivery_path = os.path.join(path, 'Q1_data.csv')
df_delivery = pd.read_csv(delivery_path)

print(f"Dataset shape: {df_delivery.shape}")

In [ ]:
# Task 2: Write your code here:
df_delivery.head()

In [ ]:
# Task 3: Write your code here:
df_delivery.info()

In [ ]:
# Task 4: Write your code here:
df_delivery.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df_delivery['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_delivery = df_delivery.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:
df_delivery = df_delivery.dropna(subset=['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs', 'Delivery_Time'])
print(f"After dropping missing values: {df_delivery.shape}")
print("Missing values remaining:", df_delivery.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df_delivery):
  duplicates = df_delivery.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_delivery.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_delivery)

In [ ]:
# Task 4: Write your code here:
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

categorical_cols = df_delivery.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    df_delivery[col] = le.fit_transform(df_delivery[col].astype(str))

df_delivery.head()

In [ ]:
# Task 5: Write your code here:

X = df_delivery.drop('Delivery_Time', axis=1)
y = df_delivery['Delivery_Time']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Task 6: Write your code here:
df_delivery['Delivery_Time'].value_counts()
#there is imbalance

In [ ]:
# Task 1: Write your code here:
# i split the x and y in previous part2 in scaling
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
print("Model trained!")
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []

for train_idx, val_idx in kfold.split(X_train_scaled):
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_fold_pred)))

mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")
print(f"RMSE: ${rmse_scores.mean():,.2f}")

In [ ]:
# Task 2,3,4,5: Write your code here:
# i did all the tasks for part3 in one cell


In [ ]:
# Task 1: Write your code here:
feature_cols = ['Distance_km','Traffic_Level'	,'Time_of_Day',	'Vehicle_Type','Preparation_Time_min','Courier_Experience_yrs','Delivery_Time']
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df_delivery['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here:
from catboost import CatBoostRegressor
models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "CatBoost": CatBoostRegressor(verbose=0)
}
all_results = {}

for name in models:
  all_results[name] = {'mse': [], 'rmse': [], 'r2': []}
  kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mse = sklearn_mse(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    # Store results
    all_results[model_name]["mse"].append(mse)
    all_results[model_name]["rmse"].append(rmse)
    all_results[model_name]["r2"].append(r2)